# Semantic Search

Have you ever wondered how on social media you can find what you are looking for, even without using the exact keywords?

That's **Semantic Search**.

All the text (and images, videos, and music) are converted into vector. When you seach for something, your query is also converted into a vector, which allows Semantic Search to measure the distance between your query and all the posts available on the social media platform, and find the ones more relevant to you.



# Vaibez semantic search V0.3
## 2026 Sep 29

Install requirements in the Python environment of your choice (`%pip install -r requirements.txt`), and restart the kernel. Place `Social_Media_Engagement_Dataset.csv` beside this notebook, then run the cells in order. The pretrained model downloads on first use; an internet connection is needed then. The CSV was not supplied here, so all results are computed locally when you run the notebook.

All project functions have type hints and Google-style docstrings with `Args` and `Returns`. Each processing step is commented.

## Project scope

We are using a dataset campaign-oriented social media prompts as a **proxy**, not as Vaibez user data (we are in a pre user phase). We retain English posts from all platforms: earlier, restricting to English Instagram posts reportedly left only around 260 records. Searchable fields are `text_content`, `hashtags`, and `topic_category`; `post_id` is retained for traceability but excluded from embedding input. We will use a pretrained neural embedding model without fine-tuning on this unlabelled corpus.

## Workflow 

For a modelling project: define the goal → EDA → split data → fit preprocessing on training data → train candidates → validate and tune → test once → visualise errors → document findings → hand off for production and monitoring. Splitting before fitted preprocessing prevents leakage.

For this project we are skipping the model train test and validation as we still we do not have Vaibez data.

# EDA

In [ ]:
# Import dependencies

# Import PyTorch on its own to prevent failure
import torch

# import all the other dependecies
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langdetect import DetectorFactory, LangDetectException, detect
from sentence_transformers import SentenceTransformer

# Display the installed PyTorch version
print(torch.__version__)

# Set a seed for repeatable language detection on ambiguous texts.
DetectorFactory.seed = 0

# Read the CSV located beside the notebook.
df = pd.read_csv("Social_Media_Engagement_Dataset.csv")

# Show the dimensions and first five records.
print(f"Rows: {len(df)}; columns: {len(df.columns)}")
display(df.head(5))

In [ ]:
# Show each original column and its pandas data type.
display(pd.DataFrame({"Column": df.columns, "Type": df.dtypes.astype(str).values}))

# Count missing values and display only affected columns.
missing_values = df.isna().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))

# Count exact duplicate rows.
print(f"Duplicate rows: {df.duplicated().sum()}")

# Print columns with repeated values, including missing values as a category.
print("Columns with repeated values:")
for column in df.columns:
    # Count unique values in the current column.
    unique_count = df[column].nunique(dropna=False)
    # Show columns that are not unique across all rows.
    if unique_count != len(df):
        print(f"{column}: {unique_count} unique values")

# Summarise numeric and non-numeric columns separately.
display(df.describe())
display(df.select_dtypes(exclude="number").describe())

In [ ]:
# Select and transpose the first record for a vertical view.
first_row_table = df.iloc[[0]].T

# Label the column holding the first row's values.
first_row_table.columns = ["Value"]

# Show one field per row.
display(first_row_table)

## Column selection for V0 semantic search

`text_content` supplies the main searchable text; `hashtags` and `topic_category` add context. `post_id` is kept for traceability, not included in embeddings. We exclude `mentions`, which was reported as the only field with missing values in the original dataset and is unnecessary for V0. Platform, location, brand and campaign fields remain outside this first retrieval experiment.

## Language filtering decision

Filtering to both English and Instagram would reportedly leave around 260 posts. For this first experiment, keep all English posts regardless of platform to retain a broader corpus. Platform is not embedded or used as a filter. Revisit the choice if the actual CSV or Vaibez content differs.

In [ ]:
# Identify the four columns selected for V0.
columns_to_keep = ["post_id", "text_content", "hashtags", "topic_category"]

# Filter to English-labelled rows and copy only the selected fields.
en_df = df.loc[df["language"] == "en", columns_to_keep].copy()

# Display the count and a preview before dropping missing rows.
print(f"English rows before missing-value removal: {len(en_df)}")
display(en_df.head(5))

## Missing data approach

For this initial proof of concept, we do not investigate imputation. Drop rows missing any of the four selected fields and verify the result. This keeps the first experiment simple; live Vaibez data may require a different policy.

In [ ]:
# Remove rows with missing values in the selected fields.
en_df = en_df.dropna().copy()

# Count remaining missing values across the selected fields.
remaining_missing = int(en_df[columns_to_keep].isna().sum().sum())

# Confirm that the corpus is non-empty and complete.
print(f"Rows remaining: {len(en_df)}")
print(f"Missing values remaining: {remaining_missing}")
print(f"All selected fields complete: {remaining_missing == 0}")
assert not en_df.empty, "No English posts remain after filtering."

# Reject blank post text, which is not represented by NaN.
assert en_df["text_content"].astype(str).str.strip().ne("").all(), "Blank post text found; review it before embedding."

## Text-length distribution

Inspect character counts as a basic text-quality distribution. Statistical outliers are prompts for manual review, not automatic deletion. Your earlier review found five plausible outliers, which should be retained if the same records appear when you rerun the notebook.

In [ ]:
# Calculate the character length of each post.
en_df["text_length"] = en_df["text_content"].str.len()

# Display descriptive statistics for post length.
display(en_df["text_length"].describe())

# Draw a histogram of text lengths.
plt.figure(figsize=(8, 4))
plt.hist(en_df["text_length"], bins=30)
plt.title("Text-length distribution")
plt.xlabel("Number of characters")
plt.ylabel("Number of posts")
plt.show()

In [ ]:
# Calculate quartiles and the interquartile range.
q1 = en_df["text_length"].quantile(0.25)
q3 = en_df["text_length"].quantile(0.75)
iqr = q3 - q1

# Set conventional IQR review boundaries.
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Display potential length outliers for manual inspection.
outliers = en_df.loc[(en_df["text_length"] < lower_bound) | (en_df["text_length"] > upper_bound)]
display(outliers[["post_id", "text_content", "text_length"]])

## Why we retain the length outliers

In the earlier run, five posts fell outside the IQR bounds. You inspected them and judged their text legitimate, so the decision was to keep them. Valid long or short posts contribute realistic variation. Check this conclusion against the current output if the source dataset changes.

# Preprocess data

In this step we pre process data to make sure it is uniform, ready to train a model (or in this case to use the pretrained model)

## Exact and near-duplicate posts

Exact duplicates are checked by text equality. For **near-duplicates**, this version compares pretrained neural embeddings of normalised post text. This may flag semantically similar posts as well as slight rewrites, so review pairs manually; the earlier count of two near-duplicates came from a different method and is **not** expected to remain identical.

In [ ]:
# Count identical text entries.
print(f"Duplicate post texts: {en_df['text_content'].duplicated().sum()}")

# Display all members of any identical-text groups.
display(en_df.loc[en_df["text_content"].duplicated(keep=False), ["post_id", "text_content"]])

In [ ]:
# Import Path to check whether the model is saved locally.
from pathlib import Path


def normalise_text(text: str) -> str:
    """Normalise a post for the near-duplicate inspection.

    Args:
        text: Original post text.

    Returns:
        Lowercase text without URLs, punctuation, or repeated whitespace.
    """
    # Convert text to lowercase.
    text = text.lower()

    # Remove URLs from the diagnostic copy.
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove punctuation from the diagnostic copy.
    text = re.sub(r"[^\w\s]", "", text)

    # Collapse repeated whitespace and trim surrounding spaces.
    return re.sub(r"\s+", " ", text).strip()


# Set the folder where the pretrained model will be saved.
model_path = Path("models/all-MiniLM-L6-v2")

# Check for the saved model's configuration file.
if (model_path / "modules.json").is_file():
    # Load the model from disk without contacting Hugging Face.
    embedding_model = SentenceTransformer(
        str(model_path),
        local_files_only=True,
    )
else:
    # Download the pretrained model when no local copy exists.
    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    # Save the model for future notebook runs and deployment.
    embedding_model.save(str(model_path))


# Create diagnostic text without modifying the searchable original.
en_df["normalised_text"] = en_df["text_content"].apply(normalise_text)

# Encode the normalised posts into unit-length neural vectors.
diagnostic_embeddings = embedding_model.encode_document(
    en_df["normalised_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# Compare every pair of posts by embedding cosine similarity.
diagnostic_similarity = diagnostic_embeddings @ diagnostic_embeddings.T

# Inspect unique pairs above the provisional similarity threshold.
near_duplicate_pairs = []

# Visit each post.
for first_index in range(len(en_df)):
    # Compare it only with later posts.
    for second_index in range(first_index + 1, len(en_df)):
        # Read the embedding similarity for this pair.
        similarity = float(
            diagnostic_similarity[first_index, second_index]
        )

        # Keep candidates for manual review.
        if similarity >= 0.90:
            near_duplicate_pairs.append(
                {
                    "first_post_id": en_df.iloc[first_index]["post_id"],
                    "second_post_id": en_df.iloc[second_index]["post_id"],
                    "similarity": similarity,
                }
            )

# Present the diagnostic candidates in a table.
near_duplicates_df = pd.DataFrame(
    near_duplicate_pairs,
    columns=["first_post_id", "second_post_id", "similarity"],
)

# Display the count and first 30 candidate pairs.
print(f"Potentially similar post pairs: {len(near_duplicates_df)}")
display(near_duplicates_df.head(30))

## Language consistency

The source marks these posts as English. Independently detect the language of `text_content` and inspect any disagreement. Language detection on short or ambiguous social-media posts can be uncertain; flags should be reviewed rather than automatically dropped.

In [ ]:
def detect_language(text: str) -> str:
    """Detect the language of a social-media post.

    Args:
        text: Post text to analyse.

    Returns:
        The detected language code, or "unknown" if detection fails.
    """
    # Attempt to detect the language of the post.
    try:
        return detect(text)
    # Return a fallback for text that cannot be classified.
    except LangDetectException:
        return "unknown"

# Detect the language of each retained post.
en_df["detected_language"] = en_df["text_content"].apply(detect_language)

# Summarise detected languages and show potential inconsistencies.
print(en_df["detected_language"].value_counts())
display(en_df.loc[en_df["detected_language"] != "en", ["post_id", "text_content", "detected_language"]])

## URLs, emojis, hashtags, mentions, and encoding issues

Inspect these features before changing the text. Hashtags and mentions can carry meaning, while broken encoding may be noise. This diagnostic examines `text_content`; the separate `hashtags` field is retained for search.

In [ ]:
def inspect_text_elements(dataframe: pd.DataFrame, text_column: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Flag selected text features and summarise their frequencies.

    Args:
        dataframe: DataFrame containing the posts.
        text_column: Name of the post-text column.

    Returns:
        A checked DataFrame and a summary with counts and percentages.
    """
    # Avoid changing the source DataFrame.
    checked_df = dataframe.copy()
    # Select the post text to inspect.
    text = checked_df[text_column].astype(str)
    # Flag URLs, emojis, inline hashtags, mentions, and common encoding artefacts.
    checked_df["has_url"] = text.str.contains(r"https?://\S+|www\.\S+", regex=True, na=False)
    checked_df["has_emoji"] = text.str.contains(r"[🌀-🫿☀-➿]", regex=True, na=False)
    checked_df["has_hashtag"] = text.str.contains(r"(?<!\w)#[\w]+", regex=True, na=False)
    checked_df["has_mention"] = text.str.contains(r"(?<!\w)@[\w]+", regex=True, na=False)
    checked_df["has_encoding_issue"] = text.str.contains(r"�|Ã.|Â.|â€™|â€œ|â€|ðŸ", regex=True, na=False)
    # Identify the generated diagnostic columns.
    check_columns = ["has_url", "has_emoji", "has_hashtag", "has_mention", "has_encoding_issue"]
    # Calculate counts and percentages of affected posts.
    summary_df = checked_df[check_columns].sum().rename_axis("check").reset_index(name="post_count")
    summary_df["percentage"] = (100 * summary_df["post_count"] / len(checked_df)).round(2)
    # Return detailed flags and the summary.
    return checked_df, summary_df

# Inspect text characteristics in the English corpus.
text_check_df, text_check_summary_df = inspect_text_elements(en_df, "text_content")

# Show the counts and percentages.
display(text_check_summary_df)

## Frequent and rare vocabulary

Word counts reveal repetitive or unusual vocabulary. Rare words may be meaningful names or niche topics, so do not automatically remove them. Your previous run found one rare word; verify the current output if the CSV has changed.

In [ ]:
def analyse_vocabulary(dataframe: pd.DataFrame, text_column: str, rare_threshold: int = 1) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Count common and rare words in the post corpus.

    Args:
        dataframe: DataFrame containing the posts.
        text_column: Name of the post-text column.
        rare_threshold: Maximum count considered rare.

    Returns:
        The full vocabulary, 30 most frequent words, and rare words.
    """
    # Extract lowercase alphabetic tokens from all post text.
    words = re.findall(r"\b[a-z]+(?:'[a-z]+)?\b", " ".join(dataframe[text_column].astype(str)).lower())
    # Count tokens and sort by descending frequency.
    vocabulary_df = pd.DataFrame(Counter(words).items(), columns=["word", "frequency"]).sort_values("frequency", ascending=False).reset_index(drop=True)
    # Return all words, frequent words, and words meeting the rarity threshold.
    return vocabulary_df, vocabulary_df.head(30).copy(), vocabulary_df.loc[vocabulary_df["frequency"] <= rare_threshold].copy()

# Analyse the post-text vocabulary.
vocabulary_df, frequent_words_df, rare_words_df = analyse_vocabulary(en_df, "text_content")

# Display the counts and the frequent and rare words.
print(f"Unique words: {len(vocabulary_df)}; rare words: {len(rare_words_df)}")
display(frequent_words_df)
display(rare_words_df)

## Topic balance and corpus coverage

Count posts per labelled topic, then examine text volume and vocabulary within each topic. This is a **lexical proxy** for coverage, not proof that all possible user search intents are represented. Query-based coverage and embedding-space coverage should be evaluated separately.

In [ ]:
# Count posts in each labelled topic.
topic_balance_df = en_df["topic_category"].value_counts().rename_axis("topic_category").reset_index(name="post_count")

# Display the distribution of topics.
display(topic_balance_df)


def analyse_corpus_coverage(dataframe: pd.DataFrame, text_column: str, topic_column: str) -> pd.DataFrame:
    """Summarise lexical coverage within each labelled topic.

    Args:
        dataframe: DataFrame containing the posts.
        text_column: Name of the post-text column.
        topic_column: Name of the topic-category column.

    Returns:
        Per-topic document, token, vocabulary, diversity, and length measures.
    """
    # Prepare one output record per labelled topic.
    results = []
    # Inspect each topic's posts independently.
    for topic, group in dataframe.groupby(topic_column):
        # Extract lowercase alphabetic words from this topic's posts.
        tokens = re.findall(r"\b[a-z]+(?:'[a-z]+)?\b", " ".join(group[text_column].astype(str)).lower())
        # Count total and distinct tokens.
        token_count = len(tokens)
        unique_count = len(set(tokens))
        # Record corpus statistics for this topic.
        results.append({
            "topic": topic,
            "document_count": len(group),
            "token_count": token_count,
            "unique_token_count": unique_count,
            "lexical_diversity": round(unique_count / token_count, 3) if token_count else 0.0,
            "average_words_per_post": round(group[text_column].astype(str).str.split().str.len().mean(), 2),
        })
    # Return a DataFrame sorted by document count.
    return pd.DataFrame(results).sort_values("document_count").reset_index(drop=True)

# Analyse topic-level lexical coverage.
corpus_coverage_df = analyse_corpus_coverage(en_df, "text_content", "topic_category")

# Display the topic-level summary.
display(corpus_coverage_df)

## Corpus coverage findings

In your earlier run, each labelled topic contained approximately 189–210 posts; average post length was about 17 words, and type–token ratios were about 0.083–0.091. These **previously reported** figures suggest repetitive campaign-oriented vocabulary, not necessarily a failure of any search model. Type–token ratio varies with corpus size. For live Vaibez, gather varied content and test queries across popular, niche, emerging, and underrepresented topics. Check the figures against the current CSV.

## If Vaibez later has labelled query–post pairs: splitting and preprocessing

This is a **future supervised fine-tuning plan**, not a step performed on the current unlabelled dataset. Store each judgement as a query, `post_id`, and relevance rating. Split by **query**, keeping all judgements for the same or near-duplicate query together: use training queries to learn, validation queries to choose settings, and untouched test queries for final evaluation. If future queries matter most, consider a later time window for testing; review near-duplicate posts so repeated content does not leak across evaluation boundaries.

EDA found complete selected fields after dropping missing rows, plausible length outliers, English text, and no observed URLs, emojis, or encoding errors. Preserve valid outliers and meaningful hashtags; normalise whitespace consistently rather than stripping useful text. If any preprocessing *learns from data*, fit it on the training portion only and apply it unchanged to validation and test. Keep test relevance judgements out of model selection. For retrieval evaluation, the searchable corpus and its document IDs must remain clearly defined separately from which query judgements are used for training or testing.

## If labelled pairs exist: training, validation, and hyperparameter tuning

Compare the **unmodified pretrained encoder** against a fine-tuned version on the same judged queries. Fine-tune using relevant query–post pairs and reviewed non-relevant examples, including difficult near-matches. On validation queries, inspect Precision@5, ranking quality such as nDCG@5, and Recall@5 **only if** relevance judgements are sufficiently complete. Examples of tunable choices: learning rate, batch size, negative sampling, or whether hashtags are included in the post representation. Review niche-query failures, false negatives, overfitting, and duplicate leakage. Freeze decisions before testing on untouched queries. No fine-tuning is performed in this notebook.

# Test Model

Test the pretrained model with the dataset

## Current PoC: pretrained semantic embeddings

We reuse `sentence-transformers/all-MiniLM-L6-v2` loaded earlier. Encode each post's `text_content`, `hashtags`, and `topic_category` together; encode queries with the same model. Both embeddings are normalised, so their dot product is cosine similarity. **Cosine distance = 1 − cosine similarity**, and smaller distance ranks first. This is meaning-based retrieval, not training a Vaibez-specific model. A distance is not a relevance judgement or a probability. The phrase “latest” does **not** apply a time filter: `timestamp` was excluded from this V0 experiment.

In [ ]:
# Verify that the expected searchable fields exist.
required_columns = {"post_id", "text_content", "hashtags", "topic_category"}
assert required_columns.issubset(en_df.columns), f"Missing columns: {required_columns - set(en_df.columns)}"

# Copy the agreed source fields, resetting the index for positional retrieval.
search_df = en_df[["post_id", "text_content", "hashtags", "topic_category"]].copy().reset_index(drop=True)

# Combine content, hashtags, and topic for embedding; never include post_id.
search_df["search_text"] = (
    search_df["text_content"].astype(str).str.strip()
    + " " + search_df["hashtags"].astype(str).str.strip()
    + " " + search_df["topic_category"].astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

# Convert each post into a unit-length neural embedding once.
post_embeddings = embedding_model.encode_document(
    search_df["search_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# Display the size of the search corpus and embedding dimension.
print(f"Indexed posts: {len(search_df)}; embedding dimensions: {post_embeddings.shape[1]}")

In [ ]:
def search_posts(query: str, top_k: int = 5) -> pd.DataFrame:
    """Rank posts using pretrained neural embeddings and cosine distance.

    Args:
        query: Search query entered by the user.
        top_k: Maximum number of ranked posts to return.

    Returns:
        A DataFrame with integer `post rank`, string `post`, and float
        `distance`, sorted from smallest cosine distance to largest.
    """
    # Reject empty queries and invalid result counts.
    if not query.strip():
        raise ValueError("query must contain text.")
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    # Encode the query using the same pretrained model as the posts.
    query_embedding = embedding_model.encode_query(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    # Compute cosine similarity to every normalised post embedding.
    similarities = post_embeddings @ query_embedding
    # Convert similarity into cosine distance; lower values rank higher.
    distances = 1.0 - similarities
    # Identify up to top_k nearest posts by ascending distance.
    ranked_indices = np.argsort(distances)[: min(top_k, len(search_df))]
    # Construct the exact three requested columns with explicit dtypes.
    results = pd.DataFrame({
        "post rank": pd.Series(range(1, len(ranked_indices) + 1), dtype="int64"),
        "post": pd.Series(search_df.iloc[ranked_indices]["text_content"].tolist(), dtype="str"),
        "distance": pd.Series([float(distances[index]) for index in ranked_indices], dtype="float64"),
    })
    # Return the ranked result table.
    return results

## Five semantic-search queries

Each `queryXdf` is a pandas DataFrame containing `post rank` (integer), `post` (text), and `distance` (float). Rank 1 has the smallest embedding cosine distance. The tables are produced when you run this notebook with your CSV; no results are invented or precomputed. **Even the closest five posts may be irrelevant** if the corpus lacks the requested topic. Manually judge relevance; distance alone is not evidence of quality.

In [ ]:
def show_wrapped_results(results: pd.DataFrame) -> None:
    """Display search results with complete, wrapped post text.

    Args:
        results: A query-results DataFrame containing a post column.

    Returns:
        None. Displays the styled DataFrame in the notebook.
    """
    # Prevent pandas from shortening long text for display
    with pd.option_context("display.max_colwidth", None):
        # Wrap text in the post column and display the table
        display(
            results.style.set_properties(
                subset=["post"],
                **{
                    "white-space": "normal",
                    "overflow-wrap": "anywhere",
                    "max-width": "600px",
                    "text-align": "left",
                },
            )
        )


In [ ]:
# Define the five exact user-specified test queries.
query1 = "latest iphone drop"
query2 = "samsung vs iphone"
query3 = "best life hack"
query4 = "Nike ads"
query5 = "Spiderman movie"

# Retrieve up to five posts for each query into the requested DataFrames.
query1df = search_posts(query1, top_k=5)
query2df = search_posts(query2, top_k=5)
query3df = search_posts(query3, top_k=5)
query4df = search_posts(query4, top_k=5)
query5df = search_posts(query5, top_k=5)

# Display each query and its corresponding ranked table.
print(f"Query 1: {query1}")
show_wrapped_results(query1df)
print(f"Query 2: {query2}")
show_wrapped_results(query2df)
print(f"Query 3: {query3}")
show_wrapped_results(query3df)
print(f"Query 4: {query4}")
show_wrapped_results(query4df)
print(f"Query 5: {query5}")
show_wrapped_results(query5df)

In [ ]:
# Embed query 1 with the same model used for the posts
query_embedding = embedding_model.encode_query(
    query1, convert_to_numpy=True, normalize_embeddings=True
)

# Calculate cosine distance to every post
distances = 1.0 - (post_embeddings @ query_embedding)

# Select the five most distant posts
bottom_indices = np.argsort(distances)[-5:][::-1]

# Display their text and distances
query1_bottom_df = pd.DataFrame({
    "post": search_df.iloc[bottom_indices]["text_content"].to_numpy(),
    "distance": distances[bottom_indices].astype(float),
})

display(query1_bottom_df)

## Interpretation and remaining evaluation

Inspect the five result tables and record human relevance judgements for each query–post pair. Precision@5 uses those judged returned posts; it cannot establish recall across the corpus without substantially more complete relevance judgements. Test both on-topic and deliberately absent topics, inspect result diversity, and investigate weak or irrelevant queries. This notebook uses **pretrained semantic retrieval**, not a Vaibez-trained model; future labelled Vaibez pairs could support fine-tuning and held-out evaluation.

# Conclusion

For “latest iphone drop”, the top result discusses an iPhone 15 but not a new release. The fifth result describes unboxing a new iPhone 15, making it more relevant to the query’s intent. The pretrained model is a reasonable V0, but these results show why relevance needs human evaluation before considering fine-tuning.

The recommenndation for Vaibez is use this V0 at launch, gather data, and mesaure how effective it serves the users. Decide whether to keep using it or train a custom model based on the findings.

# NEXT STEPS

Gather all the resources to create a docker image
1. the posts as text
1. the posts as embeddings
1. the model to convert queries into embeddings

In [ ]:
from pathlib import Path

import numpy as np

# Change this only if your notebook uses a different variable name.
posts = en_df
embeddings = np.asarray(post_embeddings)

# Check that embedding row i belongs to post row i.
if len(posts) != len(embeddings):
    raise ValueError("The number of posts and embeddings must match.")

output_dir = Path("search_data")
output_dir.mkdir(exist_ok=True)

# 1. Save the embeddings.
np.save(output_dir / "post_embeddings.npy", embeddings)

# 2. Save the posts, including their IDs and the fields used to build search text.
posts[
    ["post_id", "text_content", "hashtags", "topic_category"]
].to_csv(output_dir / "posts.csv", index=False)

print(f"Saved {len(posts)} posts and {len(embeddings)} embeddings.")

# Testing resources and process

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load the saved embeddings and posts.
data_dir = Path("search_data")
post_embeddings = np.load(data_dir / "post_embeddings.npy")
posts = pd.read_csv(
    data_dir / "posts.csv",
    dtype={"post_id": str},
    keep_default_na=False,
)

# Embedding row i must correspond to post row i.
if len(posts) != len(post_embeddings):
    raise ValueError("The number of posts and embeddings does not match.")
if len(posts) == 0:
    raise ValueError("There are no posts to search.")

# Load the already-saved model from disk.
model_path = Path("models/all-MiniLM-L6-v2")
if not (model_path / "modules.json").is_file():
    raise FileNotFoundError(f"Saved model not found at {model_path}")

embedding_model = SentenceTransformer(
    str(model_path),
    local_files_only=True,
)

# Embed the query using the same model as the posts.
query = "lates iphone drop"
query_embedding = np.asarray(embedding_model.encode(query))
post_embeddings = np.asarray(post_embeddings)

if post_embeddings.ndim != 2 or post_embeddings.shape[1] != query_embedding.size:
    raise ValueError("Post embeddings and query embedding have incompatible shapes.")

# Calculate cosine distance: 1 - cosine similarity.
query_norm = np.linalg.norm(query_embedding)
post_norms = np.linalg.norm(post_embeddings, axis=1)
if query_norm == 0 or np.any(post_norms == 0):
    raise ValueError("Cosine distance requires non-zero embeddings.")

similarities = (
    post_embeddings @ query_embedding
) / (post_norms * query_norm)
distances = 1.0 - similarities

# Sort from smallest distance to largest and print up to five posts.
top_indices = np.argsort(distances, kind="stable")[:5]

print(f'Query: "{query}"\n')

for rank, index in enumerate(top_indices, start=1):
    post = posts.iloc[index]

    print(f"{rank}. post_id: {post['post_id']}")
    print(" distance:", round(float(distances[index]), 4))
    print(f"   text_content: {post['text_content']}")
    print(f"   hashtags: {post['hashtags']}")
    print(f"   topic_category: {post['topic_category']}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: "lates iphone drop"

1. post_id: ebrkpy9w8vy4
 distance: 0.543
   text_content: How do I fix about Apple's iPhone 15? @MarketingTeam #Travel Really interested in hearing your thoughts!
   hashtags: #Travel
   topic_category: Delivery

2. post_id: z4r0dp99szmq
 distance: 0.6242
   text_content: Has anyone else experienced delivery delays with Apple's Apple Watch? Not worth the money.  #Health
   hashtags: #Health
   topic_category: Product

3. post_id: xu9qfafsw0ri
 distance: 0.6287
   text_content: Anyone have tips about Apple's iPhone 15? @RetailSupport #Premium, #Affordable Curious about your experience too.
   hashtags: #Premium, #Affordable
   topic_category: Support

4. post_id: 8ykrjkxmwvyw
 distance: 0.6411
   text_content: What's your opinion about Apple's iPhone 15? @BrandCEO, @RetailSupport #Tech, #CustomerService Would love to get your feedback!
   hashtags: #Tech, #CustomerService
   topic_category: Returns

5. post_id: 1tgb6ufvlq5i
 distance: 0.6496
   text_content: